# Desafio 4 — Pipeline Paginado: Indicadores do Banco Mundial (World Bank API)

**Conceitos praticados:** ingestão batch paginada com controle de estado, tabela de watermark,
`MERGE INTO` com chave histórica composta, tabela de fato volumosa, Lakeflow Jobs multi-task.

**Seções do exame:** S2 (Data Ingestion), S3 (Data Transformation), S4 (Lakeflow Jobs)

**API:** https://api.worldbank.org — 100% pública, sem chave, paginação explícita em JSON.

### Analogia: a Biblioteca Paginada
A API do Banco Mundial é como uma biblioteca enorme onde os livros ficam em prateleiras numeradas.
Quando você pede os dados de "população de todos os países", ela não entrega tudo de uma vez:
"Aqui estão as primeiras 100 fichas (página 1 de 12). Quando quiser mais, peça a página 2."

Seu pipeline precisa:
1. Saber quantas prateleiras existem (`pages` no metadado)
2. Percorrer todas as prateleiras em ordem
3. **Registrar onde parou** em caso de falha (tabela `ingestion_control`)
4. Retomar de onde parou na próxima execução

### Estrutura crítica da resposta (LEIA COM ATENÇÃO):
A API retorna um array JSON com **2 elementos**:
- `response[0]` → metadados de paginação: `{"page": 1, "pages": 12, "per_page": 100, "total": 1155}`
- `response[1]` → lista com os dados reais (pode ser `null` se a página não tiver dados)

```python
raw = requests.get(url).json()
meta = raw[0]        # metadados
rows = raw[1] or []  # dados reais (trata None)
total_pages = meta["pages"]
```

### 3 indicadores que vamos ingerir:
| Código | Nome | Descrição |
|---|---|---|
| `SP.POP.TOTL` | Population, total | População total por país/ano |
| `NY.GDP.MKTP.CD` | GDP (current US$) | PIB em USD corrente |
| `SP.DYN.LE00.IN` | Life expectancy at birth | Expectativa de vida ao nascer |

> ⚠️ Se o outbound internet estiver bloqueado, use o **Plano B** documentado na Célula 4.

In [0]:
%sql
-- Passo 0: criar o schema worldbank (execute uma única vez)
CREATE SCHEMA IF NOT EXISTS workshop_weather.worldbank
  COMMENT 'Indicadores socioeconômicos — World Bank API';

In [0]:
# ── Imports e configuração ──────────────────────────────────────────────────
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

import requests
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
    TimestampType, LongType
)

CATALOG = "workshop_weather"
SCHEMA  = "worldbank"
TABLE_BRONZE  = f"{CATALOG}.{SCHEMA}.raw_indicators"
TABLE_CONTROL = f"{CATALOG}.{SCHEMA}.ingestion_control"
TABLE_SILVER  = f"{CATALOG}.{SCHEMA}.fact_indicator_values"
TABLE_GOLD    = f"{CATALOG}.{SCHEMA}.gold_country_profile"

# Indicadores a ingerir
INDICATORS = {
    "SP.POP.TOTL":    "Population, total",
    "NY.GDP.MKTP.CD": "GDP (current US$)",
    "SP.DYN.LE00.IN": "Life expectancy at birth, total (years)",
}

# Códigos de agregados regionais a excluir da Gold
REGIONAL_AGGREGATES = {
    "WLD", "HIC", "LIC", "MIC", "LMC", "UMC", "ECS", "EAS", "LCN",
    "MEA", "NAC", "SAS", "SSF", "EAP", "LAC", "MNA", "SSA", "INX",
    "IBD", "IBT", "IDB", "IDX", "OED", "EMU",
}

PER_PAGE = 100

print(f"Bronze:  {TABLE_BRONZE}")
print(f"Control: {TABLE_CONTROL}")
print(f"Silver:  {TABLE_SILVER}")
print(f"Gold:    {TABLE_GOLD}")
print(f"Indicadores: {list(INDICATORS.keys())}")

### ⚠️ Plano B — API inacessível (outbound bloqueado)

Execute o script abaixo **localmente** na sua máquina. Ele coleta os 3 indicadores com paginação automática e salva como JSON.

```python
# Execute LOCALMENTE (não no Databricks):
import requests, json

def fetch_all_pages(indicator_code, per_page=100):
    all_rows, page = [], 1
    while True:
        url = (
            f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
            f"?format=json&per_page={per_page}&page={page}"
        )
        raw = requests.get(url).json()
        meta, rows = raw[0], raw[1] or []
        all_rows.extend([r for r in rows if r.get("value") is not None])
        print(f"  {indicator_code} — página {page}/{meta['pages']}, {len(rows)} registros")
        if page >= meta["pages"]:
            break
        page += 1
    return all_rows

indicadores = {
    "SP.POP.TOTL":    "worldbank_population.json",
    "NY.GDP.MKTP.CD": "worldbank_gdp.json",
    "SP.DYN.LE00.IN": "worldbank_life_expectancy.json",
}

for code, filename in indicadores.items():
    data = fetch_all_pages(code)
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"Salvo: {filename} — {len(data)} registros")
```

Faça upload dos 3 arquivos para:
**Data > Volumes > workshop_weather > bronze > landing > worldbank**

Ajuste `USE_PLAN_B = True` e os caminhos na Célula 5.

In [0]:
# ── Cria a tabela de controle de ingestão ────────────────────────────────────
# Esta tabela é o "checkpoint" do pipeline batch paginado.
# Equivale ao checkpointLocation do Structured Streaming — registra onde paramos.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_CONTROL} (
    indicator_code      STRING    NOT NULL,
    last_page_ingested  INT       NOT NULL DEFAULT 0,
    total_pages         INT,
    status              STRING    NOT NULL DEFAULT 'pending',
    last_updated        TIMESTAMP
)
USING DELTA
COMMENT 'Controle de ingestão paginada — equivalente ao checkpoint do Structured Streaming'
""")

# Inicializa registros de controle para cada indicador (se ainda não existir)
for code in INDICATORS:
    spark.sql(f"""
    MERGE INTO {TABLE_CONTROL} AS target
    USING (SELECT '{code}' AS indicator_code) AS source
    ON target.indicator_code = source.indicator_code
    WHEN NOT MATCHED THEN
        INSERT (indicator_code, last_page_ingested, total_pages, status, last_updated)
        VALUES ('{code}', 0, NULL, 'pending', current_timestamp())
    """)

print(f"[OK] Tabela de controle inicializada:")
spark.sql(f"SELECT * FROM {TABLE_CONTROL}").show(truncate=False)

In [0]:
# ── Função de ingestão paginada com controle de estado ───────────────────────

def fetch_worldbank_page(indicator_code: str, page: int, per_page: int = 100) -> tuple:
    """
    Busca uma página específica da World Bank API.
    Retorna (meta, rows) onde:
    - meta: dicionário com metadados de paginação
    - rows: lista de registros de dados (pode ser vazia)
    """
    url = (
        f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
        f"?format=json&per_page={per_page}&page={page}"
    )
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        raw = response.json()
        meta = raw[0]       # metadados de paginação
        rows = raw[1] or [] # dados reais — pode ser None, tratamos como lista vazia
        return meta, rows
    except Exception as e:
        raise RuntimeError(f"Falha na página {page} do indicador {indicator_code}: {e}")


def ingest_indicator(indicator_code: str, use_plan_b: bool = False, plan_b_path: str = None):
    """
    Ingere todas as páginas de um indicador, respeitando o estado de ingestion_control.
    - Se status = 'completed': pula (já ingerido)
    - Se status = 'in_progress' ou 'pending': retoma da last_page_ingested + 1
    - Se status = 'failed': retoma da última página bem-sucedida
    """
    ingestion_ts = datetime.utcnow()

    # Lê o estado atual deste indicador
    control_row = spark.sql(f"""
        SELECT last_page_ingested, total_pages, status
        FROM {TABLE_CONTROL}
        WHERE indicator_code = '{indicator_code}'
    """).collect()[0]

    current_status = control_row["status"]
    start_page = control_row["last_page_ingested"] + 1

    # Se já completado, não reprocessa
    if current_status == "completed":
        print(f"[SKIP] {indicator_code} já está com status=completed. Nenhuma ação.")
        return

    print(f"[INFO] Iniciando ingestão de {indicator_code} a partir da página {start_page}...")

    if use_plan_b:
        # Plano B: lê do arquivo local já com todos os dados
        print(f"[PLANO B] Lendo: {plan_b_path}")
        df_file = spark.read.format("json").load(plan_b_path)
        rows_all = [row.asDict() for row in df_file.collect()]

        # Grava tudo de uma vez na Bronze (sem paginação)
        rows_bronze = [(
            indicator_code,
            INDICATORS.get(indicator_code, ""),
            1, 1,
            json.dumps(rows_all),
            ingestion_ts
        )]
        df_b = spark.createDataFrame(rows_bronze, schema=bronze_schema)
        df_b.write.format("delta").mode("append").saveAsTable(TABLE_BRONZE)

        spark.sql(f"""
        MERGE INTO {TABLE_CONTROL} AS t
        USING (SELECT 1) AS s ON t.indicator_code = '{indicator_code}'
        WHEN MATCHED THEN UPDATE SET
            last_page_ingested = 1, total_pages = 1,
            status = 'completed', last_updated = current_timestamp()
        """)
        print(f"[PLANO B] {indicator_code} ingerido.")
        return

    # Ingestão via API com paginação
    try:
        spark.sql(f"""
        MERGE INTO {TABLE_CONTROL} AS t USING (SELECT 1) AS s
        ON t.indicator_code = '{indicator_code}'
        WHEN MATCHED THEN UPDATE SET status = 'in_progress', last_updated = current_timestamp()
        """)

        page = start_page
        total_pages = None

        while True:
            meta, rows = fetch_worldbank_page(indicator_code, page, PER_PAGE)
            total_pages = meta["pages"]

            # Filtra apenas registros com valor não nulo
            valid_rows = [r for r in rows if r.get("value") is not None]

            if valid_rows:
                rows_bronze = [(
                    indicator_code,
                    INDICATORS.get(indicator_code, ""),
                    page,
                    total_pages,
                    json.dumps(valid_rows),
                    ingestion_ts
                )]
                df_page = spark.createDataFrame(rows_bronze, schema=bronze_schema)
                df_page.write.format("delta").mode("append").saveAsTable(TABLE_BRONZE)

            # Atualiza controle de progresso
            spark.sql(f"""
            MERGE INTO {TABLE_CONTROL} AS t USING (SELECT 1) AS s
            ON t.indicator_code = '{indicator_code}'
            WHEN MATCHED THEN UPDATE SET
                last_page_ingested = {page},
                total_pages = {total_pages},
                last_updated = current_timestamp()
            """)

            print(f"  [{indicator_code}] Página {page}/{total_pages} — {len(valid_rows)} registros válidos")

            if page >= total_pages:
                break
            page += 1
            time.sleep(0.3)  # respeita rate limit da API pública

        # Marca como concluído
        spark.sql(f"""
        MERGE INTO {TABLE_CONTROL} AS t USING (SELECT 1) AS s
        ON t.indicator_code = '{indicator_code}'
        WHEN MATCHED THEN UPDATE SET
            status = 'completed', last_updated = current_timestamp()
        """)
        print(f"[OK] {indicator_code} — {total_pages} páginas ingeridas com sucesso.")

    except Exception as e:
        spark.sql(f"""
        MERGE INTO {TABLE_CONTROL} AS t USING (SELECT 1) AS s
        ON t.indicator_code = '{indicator_code}'
        WHEN MATCHED THEN UPDATE SET
            status = 'failed', last_updated = current_timestamp()
        """)
        print(f"[ERRO] {indicator_code} falhou na página {page}: {e}")
        raise

In [0]:
# ── Schema da Bronze ─────────────────────────────────────────────────────────
bronze_schema = StructType([
    StructField("indicator_code",       StringType(),    False),
    StructField("indicator_name",       StringType(),    True),
    StructField("page_number",          IntegerType(),   False),
    StructField("total_pages",          IntegerType(),   True),
    StructField("raw_payload",          StringType(),    False),  # array JSON da página como string
    StructField("_ingestion_timestamp", TimestampType(), False),
])

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_BRONZE} (
    indicator_code       STRING    NOT NULL,
    indicator_name       STRING,
    page_number          INT       NOT NULL,
    total_pages          INT,
    raw_payload          STRING    NOT NULL,
    _ingestion_timestamp TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Bronze: indicadores do Banco Mundial — 1 linha por página de resposta da API'
""")

# ── Configuração Plano B ─────────────────────────────────────────────────────
USE_PLAN_B = False
PLAN_B_PATHS = {
    "SP.POP.TOTL":    "/Volumes/workshop_weather/bronze/landing/worldbank/worldbank_population.json",
    "NY.GDP.MKTP.CD": "/Volumes/workshop_weather/bronze/landing/worldbank/worldbank_gdp.json",
    "SP.DYN.LE00.IN": "/Volumes/workshop_weather/bronze/landing/worldbank/worldbank_life_expectancy.json",
}
# ────────────────────────────────────────────────────────────────────────────

# Executa a ingestão para cada indicador
for code in INDICATORS:
    print(f"\n{'='*50}")
    print(f"Processando: {code} — {INDICATORS[code]}")
    print(f"{'='*50}")
    ingest_indicator(
        indicator_code=code,
        use_plan_b=USE_PLAN_B,
        plan_b_path=PLAN_B_PATHS.get(code)
    )

print("\n[INFO] Estado final do controle de ingestão:")
spark.sql(f"SELECT * FROM {TABLE_CONTROL}").show(truncate=False)
print("\n[INFO] Contagem da Bronze por indicador:")
spark.sql(f"SELECT indicator_code, COUNT(*) as pages, MAX(total_pages) as total_pages FROM {TABLE_BRONZE} GROUP BY 1").show()

## Passo 2 — Silver: tabela de fato `fact_indicator_values`

A Bronze armazena **1 linha por página** (o array JSON inteiro como string).
Na Silver, precisamos:
1. Ler cada linha da Bronze
2. Fazer `parse` do `raw_payload` (JSON string → array de objetos)
3. `explode()` o array → 1 linha por registro de dados
4. Extrair campos tipados (ano como INT, valor como DOUBLE)
5. Aplicar Data Quality: registros com `value = null` → `_dq_status = "null_value"`
6. `MERGE INTO` idempotente com chave `(country_code_iso3, indicator_code, year)`

In [0]:
# ── Parse Bronze → Silver ────────────────────────────────────────────────────
from pyspark.sql.functions import (
    explode, col, lit, when, from_json, get_json_object
)
from pyspark.sql.types import ArrayType

# Schema de cada registro dentro do array raw_payload
record_schema = StructType([
    StructField("indicator", StructType([
        StructField("id",    StringType(), True),
        StructField("value", StringType(), True),
    ]), True),
    StructField("country", StructType([
        StructField("id",    StringType(), True),
        StructField("value", StringType(), True),
    ]), True),
    StructField("countryiso3code", StringType(), True),
    StructField("date",            StringType(), True),  # vem como "2023" — convertemos para INT
    StructField("value",           DoubleType(), True),
])

# Lê Bronze e explode o array de registros
df_bronze_raw = spark.read.format("delta").table(TABLE_BRONZE)

df_exploded = (
    df_bronze_raw
    .withColumn("records", from_json(col("raw_payload"), ArrayType(record_schema)))
    .withColumn("record",  explode(col("records")))
    .select(
        col("record.countryiso3code").alias("country_code_iso3"),
        col("record.country.value").alias("country_name"),
        col("record.indicator.id").alias("indicator_code"),
        col("record.indicator.value").alias("indicator_name"),
        col("record.date").cast(IntegerType()).alias("year"),
        col("record.value").alias("value"),
        col("_ingestion_timestamp"),
    )
    .filter(col("country_code_iso3").isNotNull() & col("year").isNotNull())
)

# Data Quality: NULL values são preservados com flag
df_silver_staging = df_exploded.withColumn(
    "_dq_status",
    when(col("value").isNull(), lit("null_value")).otherwise(lit("valid"))
)

print("[INFO] Schema da Silver staging:")
df_silver_staging.printSchema()
print(f"\n[INFO] Total de registros: {df_silver_staging.count():,}")
print("\nDistribuição DQ:")
df_silver_staging.groupBy("_dq_status").count().show()
print("\nRegistros por indicador (válidos):")
df_silver_staging.filter(col("_dq_status") == "valid").groupBy("indicator_code").count().orderBy("indicator_code").show()

In [0]:
# ── Cria tabela Silver e executa MERGE INTO ───────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER} (
    country_code_iso3    STRING    NOT NULL,
    country_name         STRING,
    indicator_code       STRING    NOT NULL,
    indicator_name       STRING,
    year                 INT       NOT NULL,
    value                DOUBLE,
    _dq_status           STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Tabela de fato: indicadores socioeconômicos por país e ano — World Bank API'
""")

df_silver_staging.createOrReplaceTempView("staging_fact")

spark.sql(f"""
MERGE INTO {TABLE_SILVER} AS target
USING staging_fact AS source
ON  target.country_code_iso3 = source.country_code_iso3
AND target.indicator_code    = source.indicator_code
AND target.year              = source.year
WHEN MATCHED THEN UPDATE SET
    country_name         = source.country_name,
    indicator_name       = source.indicator_name,
    value                = source.value,
    _dq_status           = source._dq_status,
    _ingestion_timestamp = source._ingestion_timestamp
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[OK] MERGE INTO fact_indicator_values executado.")
spark.sql(f"SELECT COUNT(*) as total FROM {TABLE_SILVER}").show()
spark.sql(f"""
SELECT indicator_code, COUNT(*) as registros_validos
FROM {TABLE_SILVER}
WHERE _dq_status = 'valid'
GROUP BY 1 ORDER BY 1
""").show()

## Passo 3 — Gold: perfil consolidado por país

Fazemos um **pivot** dos 3 indicadores: cada indicador vira uma coluna no resultado final.
Filtramos apenas países reais (removemos agregados regionais como "World", "High income", etc.).

In [0]:
# ── Gold: gold_country_profile ────────────────────────────────────────────────
# Exclui agregados regionais (códigos na lista REGIONAL_AGGREGATES)
# e pega o ano mais recente disponível para cada par (país, indicador)

regional_list = "', '".join(REGIONAL_AGGREGATES)

df_gold = spark.sql(f"""
WITH latest_per_indicator AS (
    -- Ano mais recente com dado válido para cada par (país, indicador)
    SELECT
        country_code_iso3,
        country_name,
        indicator_code,
        value,
        year,
        ROW_NUMBER() OVER (
            PARTITION BY country_code_iso3, indicator_code
            ORDER BY year DESC
        ) AS rn
    FROM {TABLE_SILVER}
    WHERE _dq_status          = 'valid'
      AND country_code_iso3 NOT IN ('{regional_list}')
      AND LENGTH(country_code_iso3) = 3
),
pivoted AS (
    SELECT
        country_code_iso3,
        MAX(country_name) AS country_name,
        MAX(CASE WHEN indicator_code = 'SP.POP.TOTL'    AND rn = 1 THEN year  END) AS population_year,
        MAX(CASE WHEN indicator_code = 'SP.POP.TOTL'    AND rn = 1 THEN value END) AS population,
        MAX(CASE WHEN indicator_code = 'NY.GDP.MKTP.CD' AND rn = 1 THEN year  END) AS gdp_year,
        MAX(CASE WHEN indicator_code = 'NY.GDP.MKTP.CD' AND rn = 1 THEN value END) AS gdp_usd,
        MAX(CASE WHEN indicator_code = 'SP.DYN.LE00.IN' AND rn = 1 THEN year  END) AS life_exp_year,
        MAX(CASE WHEN indicator_code = 'SP.DYN.LE00.IN' AND rn = 1 THEN value END) AS life_expectancy_years
    FROM latest_per_indicator
    GROUP BY country_code_iso3
)
SELECT *
FROM pivoted
WHERE population IS NOT NULL
   OR gdp_usd     IS NOT NULL
   OR life_expectancy_years IS NOT NULL
ORDER BY population DESC NULLS LAST
""")

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_GOLD)

print(f"[OK] Gold gravada: {df_gold.count()} países")
display(spark.table(TABLE_GOLD).filter("country_code_iso3 IN ('BRA','USA','CHN','DEU','IND')"))

In [0]:
# ── Validação final ──────────────────────────────────────────────────────────
print("=" * 60)
print("VALIDAÇÃO FINAL — Desafio 4")
print("=" * 60)

bronze_n  = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_BRONZE}").collect()[0][0]
control_n = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_CONTROL} WHERE status = 'completed'").collect()[0][0]
silver_n  = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_SILVER}").collect()[0][0]
gold_n    = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_GOLD}").collect()[0][0]

print(f"Bronze (raw_indicators):          {bronze_n:,} páginas gravadas")
print(f"Controle (completed):             {control_n}/3 indicadores completos")
print(f"Silver (fact_indicator_values):   {silver_n:,} registros")
print(f"Gold   (gold_country_profile):    {gold_n} países")

print("\nBrasil no Gold:")
spark.sql(f"SELECT * FROM {TABLE_GOLD} WHERE country_code_iso3 = 'BRA'").show(truncate=False)

print("\nTop 5 países por população:")
spark.sql(f"SELECT country_name, population, gdp_usd, life_expectancy_years FROM {TABLE_GOLD} ORDER BY population DESC LIMIT 5").show(truncate=False)

## Passo 4 — Orquestração: Lakeflow Job multi-task

Agora que o pipeline funciona como notebook único, vamos orquestrá-lo com **Lakeflow Jobs**
(antigo Databricks Workflows), separando as responsabilidades em 3 tasks.

### Estrutura do Job `worldbank_pipeline`:

```
Task 1: ingest_population    → executa D4_task1_population.ipynb
    ↓
Task 2: ingest_gdp           → executa D4_task2_gdp.ipynb  (depende de Task 1)
    ↓
Task 3: process_silver_gold  → executa D4_task3_silver_gold.ipynb (depende de Tasks 1 e 2)
```

### Como criar o Job no Databricks:
1. No menu lateral: **Workflows** > **Jobs** > **Create Job**
2. Nomeie o Job: `worldbank_pipeline`
3. Task 1:
   - Nome: `ingest_population`
   - Tipo: Notebook
   - Notebook: (caminho do D4 task 1 no workspace)
   - Compute: Serverless
   - Retry on failure: 1 tentativa, delay 30s
4. Task 2: mesma config + **Depends on: Task 1**
5. Task 3: mesma config + **Depends on: Task 1 AND Task 2**

> **Por que separar em 3 tasks e não 1 notebook?**
> - Tasks 1 e 2 poderiam rodar **em paralelo** (são independentes)
> - Se a Task 1 falhar, a Task 3 não é executada (evita processar Silver incompleta)
> - Retry isolado por task: se a Task 2 falhar, a Task 1 não é reexecutada
> - Visibilidade de progresso: você vê no DAG visual qual task está em execução

In [0]:
# ── Simulação do comportamento do Lakeflow Job ───────────────────────────────
# Este código demonstra o que cada task faria se fosse um notebook separado.
# Em um Job real, cada bloco seria um notebook independente.

print("Simulação do DAG do Lakeflow Job 'worldbank_pipeline':")
print()
print("Task 1 [ingest_population] →  SP.POP.TOTL")
print("  Status: COMPLETED" if spark.sql(f"SELECT status FROM {TABLE_CONTROL} WHERE indicator_code='SP.POP.TOTL'").collect()[0][0] == 'completed' else "  Status: PENDING")
print()
print("Task 2 [ingest_gdp]         →  NY.GDP.MKTP.CD")
print("  Status: COMPLETED" if spark.sql(f"SELECT status FROM {TABLE_CONTROL} WHERE indicator_code='NY.GDP.MKTP.CD'").collect()[0][0] == 'completed' else "  Status: PENDING")
print()
print("Task 3 [process_silver_gold] → Bronze → Silver → Gold")
print(f"  Silver: {spark.sql(f'SELECT COUNT(*) as n FROM {TABLE_SILVER}').collect()[0][0]:,} registros")
print(f"  Gold:   {spark.sql(f'SELECT COUNT(*) as n FROM {TABLE_GOLD}').collect()[0][0]} países")

print("""
PARA TESTAR O RETRY DO JOB:
1. Apague manualmente 1 registro da tabela ingestion_control:
   UPDATE worldbank.ingestion_control
   SET status = 'failed', last_page_ingested = 5
   WHERE indicator_code = 'NY.GDP.MKTP.CD'

2. Reexecute o Job — a Task 2 deve retomar da página 6
3. Observe no DAG visual do Workflows que Task 1 não é reexecutada
""")

In [0]:
print("""
PERGUNTAS DE FIXAÇÃO (responda mentalmente):

1. O que a tabela `ingestion_control` e o `checkpointLocation` do
   Structured Streaming têm em comum? Qual é a diferença fundamental?

2. Por que a Bronze armazena 1 linha por PÁGINA (e não 1 linha por registro)?
   Qual é a vantagem disso para auditoria e reprocessamento?

3. Se a Task 2 falhar na página 8 de 12, o que acontece quando você
   reexecuta o Job? A Task 1 é reexecutada? A Silver fica inconsistente?

4. No pivot da Gold, usamos:
     MAX(CASE WHEN indicator_code = 'SP.POP.TOTL' AND rn=1 THEN value END)
   Por que usamos MAX() aqui se há apenas 1 linha com rn=1 por país?

5. Por que excluímos agregados regionais (WLD, HIC, etc.) da Gold?
   O que os diferencia dos países reais na estrutura da API?

6. Em qual cenário você usaria trigger(availableNow=True) com Structured Streaming
   em vez desta abordagem batch paginada?
""")